# Get local circular environment around a given point

* Often it's useful to be able to extract an environment within a given radius of a point.
* This demonstrates how to do it for a healpix grid.
* Makes use of the `haversine` function for working out the distance between two points on a spherical globe.

In [ ]:
import intake

import easygems.healpix as egh

from utils import hp_mods, haversine

In [ ]:
import cartopy.crs as ccrs
import intake
import matplotlib.pyplot as plt

import numpy as np
import healpy as hp
import xarray as xr
import easygems.healpix as egh

from utils import hp_mods, haversine

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']

In [ ]:
# Load specific model.
sim = 'um_glm_n2560_RAL3p3_tuned_hk26'
sim_cat = cat[sim]

In [ ]:
ds1h = sim_cat(zoom=8, time='PT1H').to_dask().pipe(hp_mods)

In [ ]:
# Choose a location.
lon0 = 270
lat0 = 45
# Calculate distance of all points from location.
dist_from_loc = haversine(lat0, lon0, ds1h.lat, ds1h.lon)
# Threshold based on desired radius.
radius = 1500  # km
circular_mask = dist_from_loc < radius

egh.healpix_show(circular_mask)

In [ ]:
# Select all values for rlut within radius of location.
rlut_mean = ds1h.rlut.isel(time=1, cell=circular_mask).mean()
print(rlut_mean.compute().values.item())

egh.healpix_show(circular_mask * ds1h.rlut.isel(time=1))